In [1]:
# https://platform.olimpiada-ai.ro/en/problems/218

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import string

In [2]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/dialectro/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/dialectro/test.csv")
subm = pd.read_csv('/kaggle/input/datasets/abukanabek/dialectro/sample_output.csv')

train.shape, test.shape, subm.shape

((424, 3), (107, 2), (216, 3))

In [3]:
train.head()

,ID,text,label
0,1,Atât e de înfricoșat de gândești că e turbat a...,graiul moldovenesc
1,3,Cei ce n-au dreptate n-ar mai năzui în veci la...,graiul moldovenesc
2,5,Deoarece ați primit bani de la oaspetele dumne...,graiul moldovenesc
3,7,Primiți vă rog oameni buni această mică mulțăm...,graiul moldovenesc
4,9,Fiin'c-o vint la noi vezi binie baș în zâua dă...,graiul bănățean


In [4]:
train['label'].value_counts()

label
graiul moldovenesc    183
româna standard       161
graiul bănățean        80
Name: count, dtype: int64

In [5]:
class2idx = {v: i for i, v in enumerate(train['label'].unique())}
idx2class = {v: k for k, v in class2idx.items()}

train['label'] = train['label'].map(class2idx.get)

class2idx

{'graiul moldovenesc': 0, 'graiul bănățean': 1, 'româna standard': 2}

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [7]:
train_sentences = train['text'].tolist()
test_sentences = test['text'].tolist()

train_embeddings = model.encode(train_sentences)
test_embeddings = model.encode(test_sentences)

In [8]:
from sklearn.model_selection import train_test_split

y = train['label']

train_embeddings, valid_embeddings, y_train, y_valid = train_test_split(train_embeddings, y, stratify=y, test_size=0.1)

train_embeddings.shape, valid_embeddings.shape, test_embeddings.shape

((381, 1024), (43, 1024), (107, 1024))

In [9]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    max_iter=1000,
    hidden_layer_sizes=(64)
)

model.fit(train_embeddings, y_train)

MLPClassifier(hidden_layer_sizes=64, max_iter=1000)

In [10]:
from sklearn.metrics import f1_score

y_pred = model.predict(valid_embeddings)
score = f1_score(y_valid, y_pred, average='macro')

print(f'Score: {score:.5f}')

Score: 1.00000


In [39]:
subtask1 = 0
for df in [train, test]:
    subtask1 += df['text'].map(lambda x: x.lower().count('pâni')).sum()
subtask2 = round(abs(train[train['label']==0]['text'].map(lambda x: sum([x.count(mark) for mark in string.punctuation])).mean() - train[train['label']==1]['text'].map(lambda x: sum([x.count(mark) for mark in string.punctuation])).mean()), 2)
subtask1, subtask2

(np.int64(21), np.float64(0.05))

In [41]:
diacritics = 'ă â î ș ț Ă Â Î Ș Ț'.split()

subm.iloc[:2, 2] = [subtask1, subtask2]
subm.iloc[2:2+len(test), 2] = test['text'].map(lambda x: sum([x.count(d) for d in diacritics]))
subm.iloc[2+len(test):, 2] = list(map(idx2class.get, model.predict(test_embeddings).flatten().tolist()))

subm.to_csv("submission.csv", index=False)

subm

,subtaskID,datapointID,answer
0,1,1,21
1,2,1,0.05
2,3,1491,2
3,3,1493,3
4,3,1495,3
...,...,...,...
211,4,1695,româna standard
212,4,1697,româna standard
213,4,1699,graiul bănățean
214,4,1701,româna standard
